# Profundización en Estadística para Científicos de Datos  
## Sesión 2 — Covarianza y correlación: interpretación geométrica + simulación (Colab)

**Objetivo de la sesión (práctico):**  
1) interpretar covarianza/correlación como estructura geométrica,  
2) ver por qué correlación no captura toda dependencia,  
3) estimar e interpretar $\Sigma$ en simulaciones,  
4) conectar $\Sigma$ con elipses (direcciones principales).

> Para $Z=(X,Y)^\top$,
$$
\Sigma = \mathrm{Cov}(Z)=\begin{pmatrix}
\mathrm{Var}(X) & \mathrm{Cov}(X,Y) \\
\mathrm{Cov}(Y,X) & \mathrm{Var}(Y)
\end{pmatrix}.
$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(10)


## 1. Definiciones operativas

Covarianza:
$$
\mathrm{Cov}(X,Y)=\mathbb{E}\big[(X-\mathbb{E}X)(Y-\mathbb{E}Y)\big].
$$

Correlación:
$$
\rho_{X,Y}=\frac{\mathrm{Cov}(X,Y)}{\sqrt{\mathrm{Var}(X)\,\mathrm{Var}(Y)}}.
$$

**Hecho clave:** correlación mide relación lineal y es adimensional; covarianza depende de unidades.


## 2. Simulación: normal bivariada con distintos $\rho$

Usaremos:
$$
\mu=\begin{pmatrix}0\\0\end{pmatrix},\quad
\Sigma=\begin{pmatrix}1 & \rho \\ \rho & 1\end{pmatrix}.
$$


In [ ]:
def sample_bivariate_normal(n, rho):
    mu = np.array([0.0, 0.0])
    Sigma = np.array([[1.0, rho],
                      [rho, 1.0]])
    Z = np.random.multivariate_normal(mu, Sigma, size=n)
    return Z[:,0], Z[:,1], Sigma

rhos = [-0.8, 0.0, 0.8]
n = 2000

for rho in rhos:
    X, Y, Sigma = sample_bivariate_normal(n, rho)
    plt.figure(figsize=(5,5))
    plt.scatter(X, Y, s=6, alpha=0.35)
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.title(f"Normal bivariada con rho={rho}")
    plt.axis("equal")
    plt.show()
    print("rho teórica:", rho, " | rho muestral:", np.corrcoef(X,Y)[0,1])
    print("Sigma muestral:\n", np.cov(X,Y), "\n")


## 3. Geometría: elipse de covarianza (referencia)

Diagonalizamos:
$$
\Sigma = Q\Lambda Q^\top,
$$
donde los autovalores en $\Lambda$ son varianzas en direcciones principales y $Q$ da esas direcciones.

Construimos una elipse de referencia con
$$
(z-\mu)^\top \Sigma^{-1}(z-\mu)=c.
$$


In [ ]:
def covariance_ellipse_points(mu, Sigma, c=5.991, n_points=240):
    vals, vecs = np.linalg.eigh(Sigma)
    axes = np.sqrt(c * vals)
    t = np.linspace(0, 2*np.pi, n_points)
    circle = np.vstack([np.cos(t), np.sin(t)])
    ellipse = (vecs @ (axes[:,None] * circle)) + mu[:,None]
    return ellipse[0,:], ellipse[1,:]

X, Y, _ = sample_bivariate_normal(2500, 0.8)
mu_hat = np.array([X.mean(), Y.mean()])
Sigma_hat = np.cov(X, Y)

ex, ey = covariance_ellipse_points(mu_hat, Sigma_hat)

plt.figure(figsize=(5,5))
plt.scatter(X, Y, s=6, alpha=0.25)
plt.plot(ex, ey, linewidth=2)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Nube + elipse de covarianza (referencia)")
plt.axis("equal")
plt.show()

print("mu_hat =", mu_hat)
print("Sigma_hat =\n", Sigma_hat)


## 4. Correlación 0 no implica independencia (ejemplo no lineal)

Sea $X\sim N(0,1)$ y $Y=X^2$. Hay dependencia fuerte, pero la correlación puede ser cercana a 0.


In [ ]:
n = 6000
X = np.random.randn(n)
Y = X**2

plt.figure(figsize=(5,5))
plt.scatter(X, Y, s=6, alpha=0.25)
plt.xlabel("X")
plt.ylabel("Y=X^2")
plt.title("Dependencia no lineal: correlación puede ser ~0")
plt.show()

print("Corr(X, X^2) muestral:", np.corrcoef(X, Y)[0,1])


## 5. Unidades: covarianza cambia, correlación no

Si $X' = X/1000$, entonces la covarianza con $Y$ se escala, pero la correlación no cambia.


In [ ]:
X, Y, _ = sample_bivariate_normal(5000, 0.6)

cov_xy = np.cov(X, Y)[0,1]
rho_xy = np.corrcoef(X, Y)[0,1]

X_km = X / 1000.0
cov_xkm_y = np.cov(X_km, Y)[0,1]
rho_xkm_y = np.corrcoef(X_km, Y)[0,1]

print("Cov(X,Y) =", cov_xy)
print("Cov(X/1000,Y) =", cov_xkm_y)
print("Corr(X,Y) =", rho_xy)
print("Corr(X/1000,Y) =", rho_xkm_y)


## 6. Mini-taller (en clase)

1) calcula $\mu$ y $\Sigma$ muestral,  
2) grafica la nube,  
3) grafica una elipse de covarianza de referencia,  
4) interpreta autovalores y direcciones principales.

Plantilla lista abajo (reemplaza x,y por tus columnas).


In [ ]:
# --- Plantilla: reemplaza con tu dataset ---
# import pandas as pd
# df = pd.read_csv("archivo.csv")
# x = df["col1"].to_numpy()
# y = df["col2"].to_numpy()

# Ejemplo rápido (reemplazar por datos reales)
x, y, _ = sample_bivariate_normal(2500, 0.3)

mu_hat = np.array([x.mean(), y.mean()])
Sigma_hat = np.cov(x, y)

plt.figure(figsize=(5,5))
plt.scatter(x, y, s=6, alpha=0.25)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Nube de puntos (dataset)")
plt.axis("equal")
plt.show()

ex, ey = covariance_ellipse_points(mu_hat, Sigma_hat)
plt.figure(figsize=(5,5))
plt.scatter(x, y, s=6, alpha=0.25)
plt.plot(ex, ey, linewidth=2)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Nube + elipse de covarianza (referencia)")
plt.axis("equal")
plt.show()

vals, vecs = np.linalg.eigh(Sigma_hat)
print("Sigma_hat =\n", Sigma_hat)
print("Autovalores =", vals)
print("Autovectores (columnas)=\n", vecs)


## 7. Cierre de sesión

1) distinguir covarianza vs correlación,  
2) leer $\Sigma$ como forma geométrica,  
3) construir una elipse asociada a $\Sigma$,  
4) recordar que correlación 0 no implica independencia,  
5) aplicar lo anterior en un dataset real en Colab.
